In [ ]:
import requests

#авторизация на MOEX ISS через basic-аутентификацию
MOEX_LOGIN = 'orlovapolya0211@gmail.com' #логин от moex.com
MOEX_PASSWORD = 'E!3UStXK#wAwLgh'  #пароль от moex.com

session = requests.Session()
avtoriz_otvet = session.get(
    'https://passport.moex.com/authenticate',
    auth=(MOEX_LOGIN, MOEX_PASSWORD))

if avtoriz_otvet.status_code == 200:
    print('Авторизация успешна')
    print(f'Cookie получен: {"MicexPassportCert" in session.cookies}')
else:
    print(f'Ошибка авторизации: {avtoriz_otvet.status_code}')

Авторизация успешна
Cookie получен: True


In [ ]:
#cоздаём конфигурационный файл parametrs.txt
#параметры берём из статьи Tproger
#там написано - у basicConfig() три основных параметра - level, filename, format

parametrs_content = """enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
"""
#enabled=true/false - включение/выключение logging без изменения кода
#level=INFO
#filename=logs/logging_infa - где хранятся наши логи
#Tproger: format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s
#убрали funcName потому что у нас нет функций
#%(asctime)s  - дата и время события
#%(levelname)s - уровень: INFO в нашем случае
#%(message)s  - само сообщение
#%(lineno)d - номер строки


with open('parametrs.txt', 'w') as filik:
    filik.write(parametrs_content)

print('создали parametrs.txt с такими параметрами:')
with open('parametrs.txt') as filik:
    print(filik.read())

создали parametrs.txt с такими параметрами:
enabled=true
level=INFO
filename=logs/logging_infa
format=%(asctime)s - %(levelname)s - %(lineno)d - %(message)s



In [ ]:
!mkdir -p logs #создали папку, где будет файл с логами

In [ ]:
import logging

#читаем наши настройки из parametrs.txt
with open('parametrs.txt') as filik:
    lines = filik.readlines()

enabled = lines[0].split('=')[1].strip()
level_str = lines[1].split('=')[1].strip()
filename = lines[2].split('=')[1].strip()
log_format = lines[3].split('=')[1].strip()

if enabled == 'true':

    #сбрасываем старые handlers
    #потому что иначе basicConfig игнорируется в Colab
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    #filename - из parametrs.txt
    #из Хабра нашли, что записываем сообщения в файл
    #файл будет хранить данные после завершения программы

    #format - из parametrs.txt
    #Tproger: format с %(asctime)s %(levelname)s %(message)s

    #запись в файл
    #Tproger: 'logging.basicConfig() - основная функция для настройки logging'
    #уровень INFO - вывод данных о фрагментах кода, работающих так как ожидается
    logging.basicConfig(level=logging.INFO, filename=filename, format=log_format, filemode='a')
    #filemode='a' - логи добавляются в конец файла и не стираются при перезапуске

    logging.info('Подключаем logging и собираем данные с MOEX ISS API')
    print(f'Logging работает: уровень={level_str}, файл={filename}')

else:
    print('Logging не работает')

Logging работает: уровень=INFO, файл=logs/logging_infa


In [ ]:
import requests #сначала все запросы писали через requests.get() напрямую, потом переписали через session для логгирования
import pandas as pd

Импортируем две библиотеки:
- **requests** - для отправки HTTP-запросов к API биржи (для колаба без логгирования нужна была)
- **pandas** - для работы с таблицами. Превращаем неудобный JSON в удобный DataFrame

## Запрос 1 - Дневные свечи (candles)

**Цель:** получить историю дневных цен Магнита за период 2018-01-01 - 2025-12-31.

Это главные данные всего проекта. Без них невозможно посчитать нашу таргетную переменную - изменение цены акции через 24 часа после выхода новости.

Каждая строка в ответе = один торговый день.


In [ ]:
#запрос 1: получаем дневные свечи МАГНИТ с MOEX (первые 500 строк)
response_MGNT = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MGNT/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24})

data_MGNT = response_MGNT.json()

#извлекаем данные из json
columns_MGNT = data_MGNT['candles']['columns']
rows_MGNT = data_MGNT['candles']['data']

df_MGNT = pd.DataFrame(rows_MGNT, columns=columns_MGNT)
df_MGNT['ticker'] = 'MGNT'

print(f'Count of rows: {len(df_MGNT)}')
#считаем все строки - мало ли их много, а выводит только 500
df_MGNT


Count of rows: 500


,open,close,high,low,value,volume,begin,end,ticker
0,6327.0,6485.0,6485.0,6322.0,9.100541e+08,140861,2018-01-03 00:00:00,2018-01-03 23:59:59,MGNT
1,6490.0,6629.0,6640.0,6480.0,1.125693e+09,171241,2018-01-04 00:00:00,2018-01-04 23:59:59,MGNT
2,6647.0,6724.0,6724.0,6525.0,1.507697e+09,226709,2018-01-05 00:00:00,2018-01-05 23:59:59,MGNT
3,6790.0,6600.0,6829.0,6555.0,1.794717e+09,267483,2018-01-09 00:00:00,2018-01-09 23:59:59,MGNT
4,6638.0,6660.0,6685.0,6531.0,1.390669e+09,209824,2018-01-10 00:00:00,2018-01-10 23:59:59,MGNT
...,...,...,...,...,...,...,...,...,...
495,3274.5,3265.5,3289.5,3255.5,7.489455e+08,228965,2019-12-16 00:00:00,2019-12-16 23:59:59,MGNT
496,3267.0,3272.5,3285.5,3258.5,6.973407e+08,212907,2019-12-17 00:00:00,2019-12-17 23:59:59,MGNT
497,3272.0,3271.0,3289.0,3268.5,8.305775e+08,253396,2019-12-18 00:00:00,2019-12-18 23:59:59,MGNT
498,3272.5,3276.0,3288.5,3270.5,1.072640e+09,327142,2019-12-19 00:00:00,2019-12-19 23:59:59,MGNT


**Результат:**  Count of rows: 500

Таблица содержит 9 колонок:
-  open  - цена акции Магнита в начале торгового дня (рублей за акцию)
-  close  - цена в конце торгового дня
-  high  - максимальная цена за день
-  low  - минимальная цена за день
-  value  - суммарный оборот за день в рублях (сколько рублей прошло через все сделки)
-  volume  - количество акций сменивших владельца за день
-  begin  - дата и время начала торгового дня
-  end  - дата и время конца торгового дня
-  ticker  - тикер бумаги (добавили сами, чтобы потом объединить все 7 компаний)


Также из документации (стр. 11) узнали что сервер возвращает данные порциями. Чтобы выгрузить полный объём данных, нужно последовательно делать запросы, увеличивая параметр start на значение pagesize, пока не получим пустой блок данных.



In [ ]:
logging.info('Запрос 1: проверяем есть ли данные после 500 строки (start=500)')
#проверяем есть ли данные ниже 500 строки
response_MGNT = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MGNT/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 500
      })

data_MGNT = response_MGNT.json()
print(f"Count of rows after 500 rows: {len(data_MGNT['candles']['data'])}")
#строк много - выводим оставшиеся


Count of rows after 500 rows: 500


**Результат:**  Count of rows after 500 rows: 500

HOOORRAY!!))

Ну вот видите?)

Данные не закончились на 500-й строке. Получили ещё 500 строк - это оставшиеся торговые дни, которые не влезли в первый запрос.

Теперь будем проверять, пока нам не выведет 0 значений.

In [ ]:
columns_MGNTnew = data_MGNT['candles']['columns']
rows_MGNTnew = data_MGNT['candles']['data']

df_MGNTnew = pd.DataFrame(rows_MGNTnew, columns=columns_MGNTnew)
df_MGNTnew['ticker'] = 'MGNT'

print(f'Count of rows after 500 rows: {len(df_MGNTnew)}')

#проверяем есть ли данные ниже 1000 строки
logging.info('Запрос 1: проверяем есть ли данные после 1000 строки (start=1000)')
response_MGNT2 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MGNT/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1000
      })

data_MGNT2 = response_MGNT2.json()
columns_MGNTnew2 = data_MGNT2['candles']['columns']
rows_MGNTnew2 = data_MGNT2['candles']['data']

df_MGNTnew2 = pd.DataFrame(rows_MGNTnew2, columns=columns_MGNTnew2)
df_MGNTnew2['ticker'] = 'MGNT'

print(f'Count of rows after 1000 rows: {len(df_MGNTnew2)}')

#проверяем есть ли данные ниже 1500 строки
logging.info('Запрос 1: проверяем есть ли данные после 1500 строки (start=1500)')
response_MGNT3 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MGNT/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 1500
      })

data_MGNT3 = response_MGNT3.json()
columns_MGNTnew3 = data_MGNT3['candles']['columns']
rows_MGNTnew3 = data_MGNT3['candles']['data']

df_MGNTnew3 = pd.DataFrame(rows_MGNTnew3, columns=columns_MGNTnew3)
df_MGNTnew3['ticker'] = 'MGNT'

print(f'Count of rows after 1500 rows: {len(df_MGNTnew3)}')



Count of rows after 500 rows: 500
Count of rows after 1000 rows: 500
Count of rows after 1500 rows: 500


In [ ]:
#проверяем есть ли данные ниже 2000 строки

logging.info('Запрос 1: проверяем есть ли данные после 2000 строки (start=2000)')
response_MGNT4 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MGNT/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2000
      })

data_MGNT4 = response_MGNT4.json()
columns_MGNTnew4 = data_MGNT4['candles']['columns']
rows_MGNTnew4 = data_MGNT4['candles']['data']

df_MGNTnew4 = pd.DataFrame(rows_MGNTnew4, columns=columns_MGNTnew4)
df_MGNTnew4['ticker'] = 'MGNT'

print(f'Count of rows after 2000 rows: {len(df_MGNTnew4)}')

Count of rows after 2000 rows: 73


In [ ]:
#проверяем есть ли данные ниже 2073 строки
logging.info('Запрос 1: проверяем есть ли данные после 2073 строки (start=2073)')
response_MGNT5 = session.get(
      'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MGNT/candles.json',
      params={
          'from': '2018-01-01',
          'till': '2025-12-31',
          'interval': 24,
          'start': 2073
      })

data_MGNT5 = response_MGNT5.json()
columns_MGNTnew5 = data_MGNT5['candles']['columns']
rows_MGNTnew5 = data_MGNT5['candles']['data']

df_MGNTnew5 = pd.DataFrame(rows_MGNTnew5, columns=columns_MGNTnew5)
df_MGNTnew5['ticker'] = 'MGNT'

print(f'Count of rows after 2073 rows: {len(df_MGNTnew5)}')

Count of rows after 2073 rows: 0


In [ ]:
#больше данных нет, поэтому обьединяем датафреймы в один
df_MGNTcandles = pd.concat([df_MGNT, df_MGNTnew, df_MGNTnew2, df_MGNTnew3, df_MGNTnew4], ignore_index=True)
print(f'Total rows: {len(df_MGNTcandles)}')

logging.info(f'Запрос 1: итого после склейки всех частей пяти датафреймов - {len(df_MGNTcandles)} строк')
df_MGNTcandles

Total rows: 2073


,open,close,high,low,value,volume,begin,end,ticker
0,6327.0,6485.0,6485.0,6322.0,9.100541e+08,140861,2018-01-03 00:00:00,2018-01-03 23:59:59,MGNT
1,6490.0,6629.0,6640.0,6480.0,1.125693e+09,171241,2018-01-04 00:00:00,2018-01-04 23:59:59,MGNT
2,6647.0,6724.0,6724.0,6525.0,1.507697e+09,226709,2018-01-05 00:00:00,2018-01-05 23:59:59,MGNT
3,6790.0,6600.0,6829.0,6555.0,1.794717e+09,267483,2018-01-09 00:00:00,2018-01-09 23:59:59,MGNT
4,6638.0,6660.0,6685.0,6531.0,1.390669e+09,209824,2018-01-10 00:00:00,2018-01-10 23:59:59,MGNT
...,...,...,...,...,...,...,...,...,...
2068,3004.0,3042.5,3057.5,3003.0,7.842498e+08,258556,2025-12-26 00:00:00,2025-12-26 23:59:58,MGNT
2069,3042.5,3042.5,3060.0,3033.0,1.176446e+08,38592,2025-12-27 00:00:00,2025-12-27 23:59:57,MGNT
2070,3050.0,3060.5,3064.5,3036.0,1.335406e+08,43757,2025-12-28 00:00:00,2025-12-28 23:59:57,MGNT
2071,3070.0,3017.0,3104.5,3000.0,1.328686e+09,435300,2025-12-29 00:00:00,2025-12-29 23:59:59,MGNT


Hooray!

Пустой ответ подтверждает, что мы выгрузили абсолютно все данные. После 2073 строки данных нет. Значит итоговый датафрейм  df_MGNTcandles  содержит полную историю дневных котировок Магнита за весь нужный период без пропусков.

**Результат:**
   
Total rows: 2073
   

Склеили пять кусков через  pd.concat  с параметром  ignore_index=True  - пересчитаем индексы строк заново от 0, иначе в итоговой таблице было бы пять наборов индексов, что вызвало бы у нас путаницу и отсутствие hoorray(.

Итого получили **2073 строк** - это все торговые дни Магнита за период с 01.01.2018 по 31.12.2025.



## Запрос 2 - Информация о бумаге (security info)

**Цель:** получить полную информацию по акциям Магнита: официальное название компании, международный код бумаги (ISIN), номинальная стоимость акции и уровень листинга.

Эти данные нужны нам для двух вещей:
1. Красиво описать датасет в README - чтобы было понятно с какими именно бумагами работаем
2. Добавить уровень листинга как признак в EDA - бумаги первого уровня листинга торгуются активнее и имеют более строгие требования к раскрытию информации

In [ ]:
#запрос 2: получаем статические характеристики бумаги МАГНИТ
#нам возвращается: полное название, ISIN, номинал, уровень листинга, тип бумаги
response_MGNT_infa = session.get(
    'https://iss.moex.com/iss/securities/MGNT.json')

data_MGNT_infa = response_MGNT_infa.json()

#раздел 'description' содержит характеристики в виде строк name/value
columns_MGNT_infa = data_MGNT_infa['description']['columns']
rows_MGNT_infa = data_MGNT_infa['description']['data']

df_MGNT_infa = pd.DataFrame(rows_MGNT_infa, columns=columns_MGNT_infa)
df_MGNT_infa['ticker'] = 'MGNT'

print(f'Count of rows: {len(df_MGNT_infa)}')
df_MGNT_infa

Count of rows: 27


,name,title,value,type,sort_order,is_hidden,precision,ticker
0,SECID,Код ценной бумаги,MGNT,string,1,0,NaN,MGNT
1,ISSUENAME,Наименование ценной бумаги,акции обыкновенные именные бездокументарные,string,2,0,NaN,MGNT
2,NAME,Полное наименование,"""Магнит"" ПАО ао",string,3,0,NaN,MGNT
3,SHORTNAME,Краткое наименование,Магнит ао,string,4,0,NaN,MGNT
4,ISIN,ISIN код,RU000A0JKQU8,string,5,0,NaN,MGNT
5,REGNUMBER,Номер государственной регистрации,1-01-60525-P,string,6,0,NaN,MGNT
6,ISSUESIZE,Объем выпуска,101911355,number,7,0,0.0,MGNT
7,FACEVALUE,Номинальная стоимость,0.01,number,8,0,2.0,MGNT
8,FACEUNIT,Валюта номинала,SUR,string,9,0,NaN,MGNT
9,ISSUEDATE,Дата начала торгов,2006-04-24,date,10,0,NaN,MGNT


**Результат:**  Count of rows: 27

Получили 27 строк.

Каждая строка это одна характеристика бумаги.

Каждая характеристика на отдельной строке.

Это неудобно для анализа, поэтому в следующей ячейке мы преобразуем это формат, где одна строка на компанию, каждая характеристика отдельная колонка.

In [ ]:
#преобразуем в удобный формат: одна строка - один тикер
#разворачиваем колонку 'name' в отдельные колонки через pivot
df_MGNT_infa_best = df_MGNT_infa.set_index('name')['value'].to_frame().T
df_MGNT_infa_best['ticker'] = 'MGNT'
df_MGNT_infa_best = df_MGNT_infa_best.reset_index()

print(f'Полное название: {df_MGNT_infa_best["NAME"].values[0]}')
print(f'ISIN: {df_MGNT_infa_best["ISIN"].values[0]}')
print(f'Уровень листинга: {df_MGNT_infa_best["LISTLEVEL"].values[0]}')

logging.info(f'Запрос 2: ISIN={df_MGNT_infa_best["ISIN"].values[0]}, листинг={df_MGNT_infa_best["LISTLEVEL"].values[0]}')

df_MGNT_infa_best

Полное название: "Магнит" ПАО ао
ISIN: RU000A0JKQU8
Уровень листинга: 3


name,index,SECID,ISSUENAME,NAME,SHORTNAME,ISIN,REGNUMBER,ISSUESIZE,FACEVALUE,FACEUNIT,...,MORNINGSESSION,EVENINGSESSION,WEEKENDSESSION,REGISTRY_DATE,TYPENAME,GROUP,TYPE,GROUPNAME,EMITTER_ID,ticker
0,value,MGNT,акции обыкновенные именные бездокументарные,"""Магнит"" ПАО ао",Магнит ао,RU000A0JKQU8,1-01-60525-P,101911355,0.01,SUR,...,1,1,1,2004-03-04,Акция обыкновенная,stock_shares,common_share,Акции,886,MGNT


**Результат:**
   
Полное название: "Магнит" ПАО ао
ISIN: RU000A0JKQU8
Уровень листинга: 3
   

Теперь это одна строка с колонками NAME, ISIN, LISTLEVEL и тд.

Что значат выводы:
- **NAME = «"Магнит" ПАО ао»** - официальное юридическое название компании.
- **ISIN = RU000A0JKQU8** - международный идентификационный номер ценной бумаги.
- **LISTLEVEL = 1** - первый уровень листинга на Мосбирже. Это значит, что Магнит прошёл самые строгие требования биржи по прозрачности и раскрытию информации.

**Итог запроса 2:** получили официальную информацию по бумаге.

Hooray!))

## Запрос 3 - История дивидендов (dividends)

**Цель:** получить все даты дивидендных выплат Магнита за период проекта.

Зачем нам это нужно: когда компания выплачивает дивиденды, цена акции резко падает примерно на размер дивиденда.

Если мы не пометим эти дни, то можем ошибочно решить в EDA, что резкое падение цены вызвано негативной новостью - хотя на самом деле это просто плановая выплата. Это могло бы нам помешать.

In [ ]:
logging.info('Запрос 3: загружаем историю дивидендов MGNT')
#запрос 3: получаем историю дивидендных выплат МАГНИТ
#дивиденды влияют на котировки примерно на размер дивиденда
MGNT_diva = session.get(
    'https://iss.moex.com/iss/securities/MGNT/dividends.json')

MGNT_diva_infa = MGNT_diva.json()

#извлекаем данные из json
columns_MGNT_diva = MGNT_diva_infa['dividends']['columns']
rows_MGNT_diva = MGNT_diva_infa['dividends']['data']

df_MGNT_diva = pd.DataFrame(rows_MGNT_diva, columns=columns_MGNT_diva)
df_MGNT_diva['ticker'] = 'MGNT'

print(f'Count of rows: {len(df_MGNT_diva)}')

logging.info(f'Запрос 3: получено {len(df_MGNT_diva)} дивидендных выплат')
df_MGNT_diva

Count of rows: 24


,secid,isin,registryclosedate,value,currencyid,ticker
0,MGNT,RU000A0JKQU8,2013-08-09,46.06,RUB,MGNT
1,MGNT,RU000A0JKQU8,2014-06-13,89.15,RUB,MGNT
2,MGNT,RU000A0JKQU8,2014-10-10,78.30,RUB,MGNT
3,MGNT,RU000A0JKQU8,2014-10-29,152.07,RUB,MGNT
4,MGNT,RU000A0JKQU8,2014-12-30,152.07,RUB,MGNT
5,MGNT,RU000A0JKQU8,2015-06-19,132.57,RUB,MGNT
6,MGNT,RU000A0JKQU8,2015-10-09,88.40,RUB,MGNT
7,MGNT,RU000A0JKQU8,2016-01-08,179.77,RUB,MGNT
8,MGNT,RU000A0JKQU8,2016-06-17,42.30,RUB,MGNT
9,MGNT,RU000A0JKQU8,2016-09-23,84.60,RUB,MGNT


**Результат:**  Count of rows: 24

API вернул 24 выплат дивидентов за все годы существования на бирже. Таблица содержит колонки:

- **registryclosedate** - дата закрытия реестра.
- **value** - размер дивиденда на одну акцию в рублях.

Нам нужны только выплаты за наш период 2018–2025, поэтому сейчас будет фильтровать.

In [ ]:
#фильтруем дивиденды по диапазону дат проекта 2018-01-01 - 2025-12-31
df_MGNT_diva['registryclosedate'] = pd.to_datetime(df_MGNT_diva['registryclosedate'])
df_MGNT_diva_srok = df_MGNT_diva[
    (df_MGNT_diva['registryclosedate'] >= '2018-01-01') &
    (df_MGNT_diva['registryclosedate'] <= '2025-12-31')].reset_index()

print(f'Все дивиденты за 2018-01-01 - 2025-12-31: {len(df_MGNT_diva_srok)}')

logging.info(f'Запрос 2: ISIN={df_MGNT_infa_best["ISIN"].values[0]}, листинг={df_MGNT_infa_best["LISTLEVEL"].values[0]}')
df_MGNT_diva_srok

Все дивиденты за 2018-01-01 - 2025-12-31: 11


,index,secid,isin,registryclosedate,value,currencyid,ticker
0,13,MGNT,RU000A0JKQU8,2018-07-06,135.50,RUB,MGNT
1,14,MGNT,RU000A0JKQU8,2018-12-21,137.38,RUB,MGNT
2,15,MGNT,RU000A0JKQU8,2019-06-14,166.78,RUB,MGNT
3,16,MGNT,RU000A0JKQU8,2020-01-10,147.19,RUB,MGNT
4,17,MGNT,RU000A0JKQU8,2020-06-19,157.00,RUB,MGNT
5,18,MGNT,RU000A0JKQU8,2021-01-08,245.31,RUB,MGNT
6,19,MGNT,RU000A0JKQU8,2021-06-25,245.31,RUB,MGNT
7,20,MGNT,RU000A0JKQU8,2021-12-31,294.37,RUB,MGNT
8,21,MGNT,RU000A0JKQU8,2024-01-11,412.13,RUB,MGNT
9,22,MGNT,RU000A0JKQU8,2024-07-15,412.13,RUB,MGNT


**Результат:**  Все дивиденты за 2018-01-01 - 2025-12-31: 11


Почему конвертировали дату через  pd.to_datetime : без этого Python сравнивал бы строки а не даты. Строковое сравнение работает по алфавиту и даёт неверные результаты для дат в формате YYYY-MM-DD.

**Итог запроса 3:** знаем точные даты дивидендных выплат. В EDA пометим эти дни флагом  is_dividend_day=1 .

## Запрос 4 - Текущие рыночные данные (market data)

Запрос 4 получает текущие торговые данные по Магниту - последнюю цену, объём торгов за день, количество сделок и капитализацию компании. Всё это показывает насколько активно торгуется акция прямо сейчас.


Эти данные характеризуют **ликвидность** бумаги - насколько активно она торгуется. В EDA мы будем использовать их чтобы показать: более ликвидные бумаги быстрее и точнее реагируют на новости, потому что больше участников рынка мгновенно перекладываются после публикации статьи.

In [ ]:
logging.info('Запрос 4: загружаем текущие рыночные данные MGNT')
#запрос 4: получаем текущие торговые данные МАГНИТ
response_MGNT_markdata = session.get(
    'https://iss.moex.com/iss/engines/stock/markets/shares/boards/TQBR/securities/MGNT.json')

data_MGNT_markdata = response_MGNT_markdata.json()


#раздел 'marketdata' содержит торговые данные текущей/последней сессии
columns_MGNT_markdata = data_MGNT_markdata['marketdata']['columns']
rows_MGNT_markdata = data_MGNT_markdata['marketdata']['data']

df_MGNT_markdata = pd.DataFrame(rows_MGNT_markdata, columns=columns_MGNT_markdata)
df_MGNT_markdata['ticker'] = 'MGNT'

print(f'Count of rows: {len(df_MGNT_markdata)}')

logging.info(f'Запрос 4: получено {len(df_MGNT_markdata)} строк market data')
df_MGNT_markdata

Count of rows: 1


,SECID,BOARDID,BID,BIDDEPTH,OFFER,OFFERDEPTH,SPREAD,BIDDEPTHT,OFFERDEPTHT,OPEN,...,SYSTIME,CLOSINGAUCTIONPRICE,CLOSINGAUCTIONVOLUME,ISSUECAPITALIZATION,ISSUECAPITALIZATION_UPDATETIME,ETFSETTLECURRENCY,VALTODAY_RUR,TRADINGSESSION,TRENDISSUECAPITALIZATION,ticker
0,MGNT,TQBR,3239.5,None,3240,None,0.5,76253,36800,3212.5,...,2026-03-19 23:08:33,3191,81,329683233425,23:07:55,None,846252760,2,2.293005e+09,MGNT


**Результат:**  Count of rows: 1

Одна строка - потому что это текущие торговые данные. Но колонок очень много. API возвращает абсолютно всё что знает о торговле этой бумагой прямо сейчас.

Большинство колонок нам не нужны - это технические поля биржи вроде кодов режимов торгов, флагов состояния и прочего. В следующей ячейке оставим только то что важно для нашего анализа.

In [ ]:
#оставляем только нужные колонки из market data
#используем фильтрацию через список - так код не упадёт если какой-то колонки нет в ответе
markdata_cols = ['ticker', 'LAST', 'LASTTOPREVPRICE', 'BID', 'OFFER',
            'OPEN', 'HIGH', 'LOW', 'WAPRICE', 'NUMTRADES',
            'VOLTODAY', 'VALTODAY', 'ISSUECAPITALIZATION', 'UPDATETIME']
markdata_cols = [col for col in markdata_cols if col in df_MGNT_markdata.columns]

df_MGNT_markdata_best = df_MGNT_markdata[markdata_cols].copy()

#выводим ключевые показатели
print(f'Последняя цена MGNT: {df_MGNT_markdata_best["LAST"].values[0]}')
print(f'Изменение к пред. закрытию: {df_MGNT_markdata_best["LASTTOPREVPRICE"].values[0]}')
print(f'Капитализация: {df_MGNT_markdata_best["ISSUECAPITALIZATION"].values[0]}')

logging.info(f'Запрос 4: последняя цена MGNT={df_MGNT_markdata_best["LAST"].values[0]}')

df_MGNT_markdata_best

Последняя цена MGNT: 3240
Изменение к пред. закрытию: 0.86
Капитализация: 329683233425


,ticker,LAST,LASTTOPREVPRICE,BID,OFFER,OPEN,HIGH,LOW,WAPRICE,NUMTRADES,VOLTODAY,VALTODAY,ISSUECAPITALIZATION,UPDATETIME
0,MGNT,3240,0.86,3239.5,3240,3212.5,3243,3177,3211.5,13571,263517,846252760,329683233425,22:53:32


**Результат:** (тут данные могут изменяться при каждом запуске)
   
Последняя цена MGNT: 3240
Изменение к пред. закрытию: 0.86
Капитализация: 329887056135
   

Что означают колонки которые мы оставили:
- **LAST** - последняя цена сделки (рублей за акцию)
- **LASTTOPREVPRICE** - изменение в % к цене закрытия предыдущего дня
- **BID / OFFER** - лучшая цена покупки и продажи прямо сейчас. Разница между ними называется спредом - чем он уже, тем ликвиднее бумага
- **WAPRICE** - средневзвешенная цена за весь день (учитывает объём каждой сделки)
- **NUMTRADES** - количество сделок за торговый день. У Магнита это десятки тысяч - одна из самых торгуемых бумаг на бирже
- **VOLTODAY / VALTODAY** - объём в акциях и оборот в рублях за сегодня
- **ISSUECAPITALIZATION** - рыночная капитализация в рублях.

**Итог запроса 4:** получили характеристику ликвидности Магнита на текущий момент.

## Запрос 5 - Индексная принадлежность (indices)

**Цель:** узнать в какие биржевые индексы входит Магнит.

Индексные бумаги - особая категория. Когда акция входит в IMOEX (Индекс МосБиржи), за ней автоматически следят индексные фонды (ETF) и управляющие компании. Это означает что при выходе важной новости реагируют не только частные инвесторы, но и алгоритмы фондов.

В EDA мы сравним: одинаково ли сильно реагируют на новости индексные бумаги и менее крупные компании из нашего списка.

In [ ]:
logging.info('Запрос 5: загружаем индексы MGNT')
#запрос 5: получаем список биржевых индексов в которые входит МАГНИТ
response_MGNT_idishka = session.get(
    'https://iss.moex.com/iss/securities/MGNT/indices.json')

data_MGNT_idishka = response_MGNT_idishka.json()

#извлекаем данные из json
columns_MGNT_idishka = data_MGNT_idishka['indices']['columns']
rows_MGNT_idishka = data_MGNT_idishka['indices']['data']

df_MGNT_idishka = pd.DataFrame(rows_MGNT_idishka, columns=columns_MGNT_idishka)
df_MGNT_idishka['ticker'] = 'MGNT'

print(f'Count of rows: {len(df_MGNT_idishka)}')

logging.info(f'Запрос 5: MGNT входит в {len(df_MGNT_idishka)} индексов')

df_MGNT_idishka

Count of rows: 18


,SECID,SHORTNAME,FROM,TILL,CURRENTVALUE,LASTCHANGEPRC,LASTCHANGE,ticker
0,EPSI,Субиндекс акций,2010-12-16,2025-03-20,1665.38,0.00,-0.02,MGNT
1,HCAP,HCAP,2024-06-21,2024-09-26,907.67,-0.07,-0.66,MGNT
2,ICLIMATE,Индекс МосБиржи Климатический,2025-07-01,2026-03-18,1030.83,0.23,2.39,MGNT
3,IMOEXCNY,Индекс МосБиржи в юанях,2024-09-24,2025-03-20,1060.38,-2.49,-27.11,MGNT
4,IMOEXW,IMOEXW,2023-01-16,2025-03-20,2881.58,-0.15,-4.34,MGNT
5,MOEXBC,Индекс голубых фишек,2013-06-18,2024-12-19,19057.56,-0.03,-5.43,MGNT
6,MRRT,Индекс MRRT,2021-01-15,2026-03-18,2335.80,0.05,1.15,MGNT
7,MRSV,Индекс MRSV,2021-01-15,2026-03-18,2173.86,-0.22,-4.80,MGNT
8,MRSVR,Индекс МосБиржи-РСПП MRSV RU Co,2021-03-25,2026-03-18,2237.45,-0.22,-4.93,MGNT
9,RTScr,Индекс РТС потреб. сектора,2009-09-30,2026-03-19,190.62,-2.72,-5.33,MGNT


**Результат:**  Count of rows: 18


Важные колонки:
- **SECID** - код индекса
- **FROM / TILL** - период с которого по который бумага входит в индекс
- **CURRENTVALUE** - текущее значение самого индекса

In [ ]:
#проверяем входит ли MGNT в IMOEX (главный российский фондовый индекс)
if len(df_MGNT_idishka) > 0:
    imoex_check = df_MGNT_idishka[df_MGNT_idishka['SECID'] == 'IMOEX']
    print(f'MGNT входит в IMOEX: {len(imoex_check) > 0}')
else:
    print('MGNT не входит ни в один индекс MOEX')
logging.info(f'Запрос 5 завершили - все данные по MGNT собраны')


MGNT входит в IMOEX: True


**Результат:**
   
MGNT входит в IMOEX: True
   

Магнит входит в IMOEX - это подтверждает что он является влиятельной компанией российского рынка.

**Итог запроса 5:** Магнит входит в 18 индексов включая главный IMOEX. В EDA это будет важным признаком при анализе силы реакции на новости.



